# Analyse Exploratoire — Dakar Power Prediction

Exploration du dataset `synthetic_data_v2.csv` : distributions, corrélations, patterns temporels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
df = pd.read_csv('../data/synthetic/synthetic_data_v2.csv', parse_dates=['date_heure'])

print('Shape   :', df.shape)
print('\ndtypes :')
print(df.dtypes)
print('\nMissing :')
print(df.isnull().sum())
df.describe()

In [ ]:
taux = df.groupby('quartier')['coupure'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#dc3545' if t >= 10 else '#ffc107' if t >= 7 else '#28a745' for t in taux]
taux.plot.bar(ax=ax, color=colors, edgecolor='white')
ax.set_title('Taux de Coupure par Quartier (%)')
ax.set_xlabel('Quartier')
ax.set_ylabel('Taux (%)')
ax.tick_params(axis='x', rotation=30)
for i, v in enumerate(taux):
    ax.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ['temp_celsius', 'humidite_percent', 'vitesse_vent', 'conso_megawatt',
            'heure', 'jour_semaine', 'mois', 'saison', 'is_peak_hour', 'coupure']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, ax=ax, linewidths=0.5
)
ax.set_title('Heatmap de Corrélation des Features')
plt.tight_layout()
plt.show()

In [ ]:
by_hour = df.groupby('heure')['coupure'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(by_hour.index, by_hour.values, marker='o', color='steelblue', linewidth=2)
ax.fill_between(by_hour.index, by_hour.values, alpha=0.15, color='steelblue')
ax.axvspan(18, 22, alpha=0.12, color='red', label='Heures de pointe (18-22h)')
ax.set_title('Taux de Coupure par Heure de la Journée')
ax.set_xlabel('Heure')
ax.set_ylabel('Taux (%)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
by_month = df.groupby('mois')['coupure'].mean() * 100
month_labels = ['Jan','Fév','Mar','Avr','Mai','Juin','Juil','Aoû','Sep','Oct','Nov','Déc']

fig, ax = plt.subplots(figsize=(10, 4))
by_month.plot.bar(ax=ax, color='coral', edgecolor='white')
ax.set_title('Taux de Coupure par Mois')
ax.set_xlabel('Mois')
ax.set_ylabel('Taux (%)')
ax.set_xticklabels(month_labels, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['coupure_label'] = df['coupure'].map({0: 'Pas de coupure', 1: 'Coupure'})

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df, x='coupure_label', y='temp_celsius',
    palette={'Pas de coupure': '#28a745', 'Coupure': '#dc3545'},
    ax=ax
)
ax.set_title('Distribution de la Température selon la Coupure')
ax.set_xlabel('')
ax.set_ylabel('Température (°C)')
plt.tight_layout()
plt.show()

df.drop(columns=['coupure_label'], inplace=True)